# Cost of PartiSet and of the Baseline: fusion stage, participant grouping and pre-processing per clip

All numbers are measured in this session on the GPU shown in the first cell, so that they can be reported side by side:

1. **Fusion stage** (precomputed features): parameters, FLOPs per sample, forward and forward+backward time per sample, peak memory, for PartiSet and the Baseline.
2. **Participant grouping** of PartiSet (clustering of the faces of a sample, slot assignment, turn descriptors), per sample.
3. **Pre-processing per clip** with the extraction code of the experiments (the cells are copied verbatim from `g8a_role_features`): video decoding, face detection + ArcFace + HSEmotion, audio + ECAPA.
4. **Unit costs of CLIP** (16 images) and of the YOLOv8-small person detector (16 frames), used by the Baseline's body branch. AudioCLIP is not timed here.

Attach: the code dataset, the Hi-EF dataset, the split table and the participant-feature shards. GPU on, Internet on (model weights are downloaded).

In [ ]:
import glob, os, shutil, subprocess, sys, time, json
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none', '| torch', torch.__version__)
CODE_SRC = next(p for p in glob.glob('/kaggle/input/**/partiset', recursive=True) if os.path.isdir(p))
CODE_ROOT = '/kaggle/working/code'
shutil.copytree(os.path.dirname(CODE_SRC), CODE_ROOT, dirs_exist_ok=True)
sys.path.insert(0, CODE_ROOT)

In [ ]:
# ======== CONFIG (the constants of g8a_role_features, with the paths found automatically) ========
import glob, os
def find(pattern, d=False):
    hits = [p for p in sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True)) if (os.path.isdir(p) if d else True)]
    assert hits, f'attach the dataset holding {pattern}'
    return hits[0]
DATASET_DIR = find('hi-ef-dataset', d=True)
SPLIT_CSV = find('source_folder_split_seed42.csv')
OUT_DIR = "/kaggle/working"
SHARD_DIR = os.path.dirname(find('shard_*.pkl'))
SAMPLE_FPS, MAX_FRAMES = 4, 32
DET_SIZE = (640, 640)
MIN_DET_SCORE, MIN_FACE_PX = 0.5, 24
AUDIO_SR = 16000
WIN_S, HOP_S = 1.5, 0.75
ENV_HZ = 10
N_DECODE_THREADS = 4
FER_URL = ("https://github.com/HSE-asavchenko/face-emotion-recognition/raw/main/models/affectnet_emotions/onnx/"
           "enet_b0_8_va_mtl.onnx")
N_CLIPS, N_SAMPLES = 40, 200      # clips timed per stage, samples used to time the grouping

## 1. Fusion stage

In [ ]:
import torch.nn as nn, torch.nn.functional as F
from torch.utils.flop_counter import FlopCounterMode
from partiset import BASE, Batch, PartiSet
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
B, FD, MAXF, REPEATS = 64, 146, 24, 30

# Baseline classes: copied from the baseline notebook, executed on synthetic features of the real shapes
nb = json.load(open(f'{CODE_ROOT}/notebooks/baseline_vit.ipynb'))
src = next(''.join(c['source']) for c in nb['cells'] if c['cell_type'] == 'code' and 'class Baseline' in ''.join(c['source']))
FEAT = {'face': torch.randn(3 * B, 16, 512, device=dev), 'ori': torch.randn(3 * B, 16, 512, device=dev),
        'body': torch.randn(3 * B, 16, 512, device=dev), 'text': torch.randn(3 * B, 512, device=dev), 'audio': torch.randn(3 * B, 527, device=dev)}
CLIPIDX = torch.arange(3 * B, device=dev).reshape(B, 3)
def gather(idx):
    flat = idx.reshape(-1)
    return {k: v[flat].reshape(*idx.shape, *v.shape[1:]) for k, v in FEAT.items()}
ns = dict(torch=torch, nn=nn, F=F, FEAT=FEAT, CLIPIDX=CLIPIDX, gather=gather)
import ast
exec(compile(ast.Module([n for n in ast.parse(src).body if isinstance(n, ast.ClassDef)], []), 'baseline', 'exec'), ns)    # classes only
baseline = ns['Baseline'](True, True, 'trans', 'lstm').to(dev)       # text + audio, Transformer intra-clip, LSTM+Transformer inter-clip
ix = torch.arange(B, device=dev); yb = torch.randint(0, 7, (B,), device=dev)

b = Batch(torch.randn(B, 3, 3, MAXF, FD, device=dev), torch.randn(B, 1, 3, 32, FD, device=dev),
          torch.rand(B, 3, 3, MAXF, device=dev) > 0.5, torch.rand(B, 1, 3, 32, device=dev) > 0.5,
          torch.randn(B, 3, 512, device=dev), F.normalize(torch.randn(B, 3, 527, device=dev), dim=-1), torch.ones(B, 3, device=dev),
          torch.randn(B, 3, 1024, device=dev), torch.randn(B, 3, 9, device=dev), yb, yb)
partiset = PartiSet(BASE, FD, 9).to(dev)

def timed(fn):
    for _ in range(3): fn()
    if dev == 'cuda': torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    t = []
    for _ in range(REPEATS):
        t0 = time.perf_counter(); fn()
        if dev == 'cuda': torch.cuda.synchronize()
        t.append((time.perf_counter() - t0) / B * 1000)
    t.sort(); return t[len(t) // 2], t[int(0.95 * (len(t) - 1))]

def profile(name, model, fwd, train_loss):
    n = sum(p.numel() for p in model.parameters())
    # FLOPs are counted in train mode: in eval mode nn.TransformerEncoder uses a fused kernel that the FLOP counter does not see
    model.train()
    with FlopCounterMode(display=False) as fc: fwd()
    flops = fc.get_total_flops() / B
    model.eval()
    with torch.no_grad():
        f = timed(fwd)
    if dev == 'cuda': fm = torch.cuda.max_memory_allocated() / 2**20
    model.train()
    def step():
        train_loss().backward(); model.zero_grad()
    t = timed(step)
    mem = torch.cuda.max_memory_allocated() / 2**20 if dev == 'cuda' else float('nan')
    return dict(model=name, params_M=n / 1e6, GFLOPs_per_sample=flops / 1e9, fwd_ms_median=f[0], fwd_ms_p95=f[1], train_ms_median=t[0], train_ms_p95=t[1], peak_MiB_train=mem)

def ps_loss():
    logits, aux = partiset(b, train=True)
    return F.cross_entropy(logits, yb) + sum(w * F.cross_entropy(l, t.clamp(min=0)) for l, t, w in aux.values())
rows = [profile('Baseline (F,B,S,T,A; Trans, LSTM+Trans)', baseline, lambda: baseline(ix), lambda: F.cross_entropy(baseline(ix), yb)),
        profile('PartiSet', partiset, lambda: partiset(b), ps_loss)]
# PartiSet: the FLOPs depend on the number of face frames kept per (slot, turn) cell
for mf in (4, 24):
    bb = Batch(torch.randn(B, 3, 3, mf, FD, device=dev), b.pool, torch.rand(B, 3, 3, mf, device=dev) > 0.5, *b[3:])
    partiset.train()
    with FlopCounterMode(display=False) as fc: partiset(bb)
    print(f'PartiSet, {mf} frames per cell: {fc.get_total_flops() / B / 1e6:.1f} MFLOPs per sample')
import pandas as pd
FUSION = pd.DataFrame(rows); pd.set_option('display.width', 220)
print(FUSION.round(3).to_string(index=False)); FUSION.to_csv('/kaggle/working/cost_fusion.csv', index=False)

## 2. Participant grouping (PartiSet), per sample

In [ ]:
import numpy as np
from partiset.data import load_split
from partiset.participants import ParticipantBuilder, load_feature_shards
_, SP = load_split(DATASET_DIR, SPLIT_CSV)
SP = SP[SP.split != 'test'].reset_index(drop=True)
sub = SP.sample(N_SAMPLES, random_state=0).reset_index(drop=True)
t0 = time.time(); SH = load_feature_shards(SHARD_DIR); t_load = time.time() - t0
t0 = time.time(); PB = ParticipantBuilder(sub, SH); t_group = time.time() - t0
clips_sub = sorted(set(sub[['clip1', 'clip2', 'clip3']].values.ravel()))
t0 = time.time(); PB.build_face_tensors(clips_sub, 'cpu'); t_tensor = time.time() - t0
GROUP = dict(samples=N_SAMPLES, load_shards_s=t_load, grouping_ms_per_sample=t_group / N_SAMPLES * 1000, face_tensors_ms_per_sample=t_tensor / N_SAMPLES * 1000)
print(GROUP); json.dump(GROUP, open('/kaggle/working/cost_grouping.json', 'w'))

## 3. Pre-processing per clip (extraction code of the experiments, copied verbatim)

In [ ]:
!pip install -q insightface speechbrain onnx
!pip uninstall -y -q onnxruntime onnxruntime-gpu
!pip install -q "onnxruntime-gpu==1.22.0"

In [ ]:
import os, glob, pickle, time, urllib.request
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, cv2
import torch
import onnxruntime as ort
if hasattr(ort, 'preload_dlls'):
    try:
        ort.preload_dlls()
    except Exception as e:
        print("preload_dlls:", e)
PROV = ['CUDAExecutionProvider', 'CPUExecutionProvider']

roots = sorted(glob.glob(os.path.join(DATASET_DIR, "*", "Hi-EF")))
VIDEO_ROOTS = [os.path.join(r, "video") for r in roots if os.path.isdir(os.path.join(r, "video"))]
AUDIO_ROOTS = [os.path.join(r, "audio") for r in roots if os.path.isdir(os.path.join(r, "audio"))]


def find_media(roots_, clip, exts):
    ep, num = clip.split('/')
    for root in roots_:
        for ext in exts:
            p = os.path.join(root, ep, num + ext)
            if os.path.exists(p):
                return p
    return None


sp = pd.read_csv(SPLIT_CSV, dtype=str, usecols=['sample_id', 'split', 'clip1', 'clip2', 'clip3'])   # no label columns
clips = sorted(set(sp[['clip1', 'clip2', 'clip3']].values.ravel()))
print(f"MCIS {len(sp)} {sp.split.value_counts().to_dict()} | clips I-III {len(clips)} | "
      f"missing video {sum(find_media(VIDEO_ROOTS, c, ('.mp4',)) is None for c in clips)} | "
      f"missing audio {sum(find_media(AUDIO_ROOTS, c, ('.mp3', '.wav')) is None for c in clips)}")

In [ ]:
from insightface.app import FaceAnalysis
app = FaceAnalysis(name='buffalo_l', allowed_modules=['detection', 'recognition', 'landmark_3d_68'], providers=PROV)
app.prepare(ctx_id=0, det_size=DET_SIZE)
print("insightface providers:", {k: m.session.get_providers() for k, m in app.models.items()})

fer_path = f"{OUT_DIR}/enet_b0_8_va_mtl.onnx"
if not os.path.exists(fer_path) or os.path.getsize(fer_path) < 1_000_000:   # re-fetch missing / truncated files
    urllib.request.urlretrieve(FER_URL, fer_path)
assert os.path.getsize(fer_path) > 1_000_000, f"FER model download looks broken ({os.path.getsize(fer_path)} bytes)"
# expose the penultimate (pooled) features as a second output: input of the last Gemm/MatMul
fer_emb_path = f"{OUT_DIR}/enet_b0_8_va_mtl_emb.onnx"
try:
    import onnx
    m = onnx.load(fer_path)
    last = [n for n in m.graph.node if n.op_type in ('Gemm', 'MatMul')][-1]
    m.graph.output.append(onnx.helper.make_tensor_value_info(last.input[0], onnx.TensorProto.FLOAT, None))
    onnx.save(m, fer_emb_path)
    fer = ort.InferenceSession(fer_emb_path, providers=PROV)
except Exception as e:
    print("could not expose the FER embedding, logits only:", repr(e))
    fer = ort.InferenceSession(fer_path, providers=PROV)
FER_IN = fer.get_inputs()[0].name
print("FER providers:", fer.get_providers(), "| outputs", [(o.name, o.shape) for o in fer.get_outputs()])
MEAN, STD = np.array([0.485, 0.456, 0.406], np.float32), np.array([0.229, 0.224, 0.225], np.float32)


def fer_batch(crops):
    if not crops:
        return np.zeros((0, 10), np.float32), None
    x = np.stack([(cv2.cvtColor(cv2.resize(c, (224, 224)), cv2.COLOR_BGR2RGB).astype(np.float32) / 255 - MEAN) / STD
                  for c in crops]).transpose(0, 3, 1, 2).astype(np.float32)
    outs = fer.run(None, {FER_IN: x})
    head = outs[0][:, :10] if outs[0].shape[1] >= 10 else np.pad(outs[0], ((0, 0), (0, 10 - outs[0].shape[1])))
    emb = outs[1].reshape(len(x), -1).astype(np.float16) if len(outs) > 1 else None
    return head.astype(np.float32), emb


def read_frames(path):
    # sequential decode, keep ~SAMPLE_FPS frames per second (<= MAX_FRAMES, evenly spread)
    cap = cv2.VideoCapture(path)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    want = max(1, min(MAX_FRAMES, int(round(n / fps * SAMPLE_FPS)))) if n else MAX_FRAMES
    keep = set(np.linspace(0, max(n - 1, 0), want).astype(int).tolist())
    frames, times, i = [], [], 0
    while True:
        ok = cap.grab()
        if not ok:
            break
        if i in keep:
            ok, fr = cap.retrieve()
            if ok:
                frames.append(fr); times.append(i / fps)
        i += 1
        if i > max(keep, default=0):
            break
    cap.release()
    return frames, times, {'n_video_frames': n, 'fps': fps, 'duration': n / fps if fps else 0.0}


def analyse_frames(frames, times):
    dets, crops = [], []
    for fi, fr in enumerate(frames):
        H, W = fr.shape[:2]
        for f in app.get(fr):
            x1, y1, x2, y2 = [float(v) for v in f.bbox]
            if f.det_score < MIN_DET_SCORE or min(x2 - x1, y2 - y1) < MIN_FACE_PX:
                continue
            m = 0.1 * max(x2 - x1, y2 - y1)
            crops.append(fr[int(max(0, y1 - m)):int(min(H, y2 + m)), int(max(0, x1 - m)):int(min(W, x2 + m))])
            lm = getattr(f, 'landmark_3d_68', None)
            pose = getattr(f, 'pose', None)
            d = {'frame': fi, 't': times[fi], 'box': np.array([x1 / W, y1 / H, x2 / W, y2 / H], np.float32),
                 'score': float(f.det_score), 'pose': np.asarray(pose if pose is not None else [0, 0, 0], np.float32),
                 'arc': f.normed_embedding.astype(np.float16)}
            if lm is not None:
                bh = max(y2 - y1, 1.0)
                d['mouth'] = float(np.linalg.norm(lm[66, :2] - lm[62, :2]) / bh)
                d['lm'] = np.stack([(lm[:, 0] - x1) / max(x2 - x1, 1.0), (lm[:, 1] - y1) / bh], 1).astype(np.float16)
            else:
                d['mouth'] = np.nan
            dets.append(d)
    head, emb = fer_batch(crops)
    for k, d in enumerate(dets):
        d['fer'] = head[k]
        if emb is not None:
            d['fer_emb'] = emb[k]
    return dets

In [ ]:
import librosa
try:
    from speechbrain.inference.speaker import EncoderClassifier
except ImportError:
    from speechbrain.pretrained import EncoderClassifier
spk = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir=f"{OUT_DIR}/ecapa",
                                     run_opts={"device": "cuda" if torch.cuda.is_available() else "cpu"})


def ecapa(wav):
    with torch.no_grad():
        e = spk.encode_batch(torch.tensor(wav, dtype=torch.float32).unsqueeze(0)).reshape(-1).cpu().numpy()
    return (e / (np.linalg.norm(e) + 1e-9)).astype(np.float16)


def analyse_audio(path):
    if path is None:
        return None
    try:
        wav, _ = librosa.load(path, sr=AUDIO_SR, mono=True)
    except Exception:
        return None
    out = {'duration': len(wav) / AUDIO_SR}
    hop = AUDIO_SR // ENV_HZ
    out['env'] = np.array([np.sqrt(np.mean(wav[i:i + hop] ** 2)) if len(wav[i:i + hop]) else 0.0
                           for i in range(0, len(wav), hop)], np.float32)
    out['ecapa'] = ecapa(wav) if len(wav) >= AUDIO_SR // 2 else None
    win, step = int(WIN_S * AUDIO_SR), int(HOP_S * AUDIO_SR)
    starts = list(range(0, max(len(wav) - win, 0) + 1, step)) if len(wav) >= win else []
    out['win_t'] = np.array([s / AUDIO_SR for s in starts], np.float32)
    out['win_ecapa'] = np.stack([ecapa(wav[s:s + win]) for s in starts]) if starts else np.zeros((0, 192), np.float16)
    return out

In [ ]:
# decode / face detection + ArcFace + HSEmotion / audio + ECAPA, per clip (GPU synchronised)
rng = np.random.default_rng(0)
sel = list(rng.choice(clips, N_CLIPS, replace=False))
def sync():
    if torch.cuda.is_available(): torch.cuda.synchronize()
_ = read_frames(find_media(VIDEO_ROOTS, sel[0], ('.mp4', '.avi', '.mkv', '.mov')))      # warm-up
_ = analyse_frames(*_[:2]) if False else None
T = {'decode': [], 'faces': [], 'audio': []}; FRAMES = []
for c in sel:
    pv = find_media(VIDEO_ROOTS, c, ('.mp4', '.avi', '.mkv', '.mov')); pa = find_media(AUDIO_ROOTS, c, ('.mp3', '.wav', '.flac', '.m4a'))
    t0 = time.perf_counter(); frames, times, meta = read_frames(pv); T['decode'].append(time.perf_counter() - t0)
    t0 = time.perf_counter(); faces = analyse_frames(frames, times); sync(); T['faces'].append(time.perf_counter() - t0)
    t0 = time.perf_counter(); au = analyse_audio(pa); sync(); T['audio'].append(time.perf_counter() - t0)
    FRAMES.append(frames)
EXTRACT = pd.DataFrame({k: pd.Series(v).describe(percentiles=[.5, .95])[['mean', '50%', '95%']] for k, v in T.items()}).T
EXTRACT['total_clip_s'] = EXTRACT['mean']
print(EXTRACT.round(3).to_string())
tot = np.array(T['decode']) + np.array(T['faces']) + np.array(T['audio'])
print(f'per clip: mean {tot.mean():.2f} s, median {np.median(tot):.2f} s, p95 {np.percentile(tot, 95):.2f} s | '
      f'clips of the development and test tables: {len(clips)} -> about {tot.mean() * len(clips) / 3600:.1f} GPU-hours at the mean')
EXTRACT.to_csv('/kaggle/working/cost_extraction_per_clip.csv')

## 4. Unit costs of CLIP (16 images) and of the YOLOv8-small person detector (16 frames)

In [ ]:
!pip install -q ultralytics transformers
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
dev2 = 'cuda' if torch.cuda.is_available() else 'cpu'
clip = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(dev2).eval().half() if dev2 == 'cuda' else CLIPModel.from_pretrained('openai/clip-vit-base-patch32').eval()
proc = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
imgs = [Image.fromarray(f[:, :, ::-1]) for f in next(fr for fr in FRAMES if len(fr) >= 16)[:16]]
def clip_img():
    x = proc(images=imgs, return_tensors='pt')['pixel_values'].to(dev2)
    with torch.no_grad(): clip.get_image_features(pixel_values=x.half() if dev2 == 'cuda' else x)
    sync()
from ultralytics import YOLO
yolo = YOLO('yolov8s.pt')
frames16 = [np.array(i)[:, :, ::-1] for i in imgs]
def yolo16():
    yolo.predict(frames16, classes=[0], conf=0.4, verbose=False); sync()
def med(fn, n=15):
    fn(); fn(); t = []
    for _ in range(n):
        t0 = time.perf_counter(); fn(); t.append(time.perf_counter() - t0)
    return float(np.median(t))
UNIT = dict(clip_16_images_s=med(clip_img), yolov8s_16_frames_s=med(yolo16))
print(UNIT); json.dump(UNIT, open('/kaggle/working/cost_units.json', 'w'))

## Summary for the paper

Report next to the fusion table: the GPU, the fusion rows of section 1, the grouping time per sample (section 2), the per-clip pre-processing (section 3) and the CLIP / YOLO unit costs (section 4).
The Baseline's body branch needs the YOLOv8-small detector and CLIP on person crops; PartiSet does not use body features. AudioCLIP features (both models) are not timed here.